### Stage 3: Model Trainer

In [1]:
import os
os.chdir("../")
print('Working directory:', os.getcwd())

Working directory: c:\Users\HP\Text-Summarizer


### 1. Config Entity

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    transformed_train_dir: Path
    transformed_test_dir: Path
    model_name: str
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: int
    gradient_accumulation_steps: int

### 3. Configuration Manager

In [4]:
from src.textsummarizer.utils.common import read_yaml
from src.textsummarizer.utils.common import ConfigBox

CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")

class ConfigurationManager:
    def __init__(self):
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.params = read_yaml(PARAMS_FILE_PATH)

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        cfg = self.config.artifacts.model_trainer
        trans = self.config.artifacts.data_transformation
        params = self.params.model_trainer
        os.makedirs(cfg.root_dir, exist_ok=True)
        return ModelTrainerConfig(
            root_dir=Path(cfg.root_dir),
            transformed_train_dir=Path(trans.root_dir) / 'transformed_train',
            transformed_test_dir=Path(trans.root_dir) / 'transformed_test',
            model_name=params.model_name,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            per_device_eval_batch_size=params.per_device_eval_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            evaluation_strategy=params.evaluation_strategy,
            eval_steps=params.eval_steps,
            save_steps=params.save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps
        )

### Model Trainer Component

In [5]:
import torch 
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f'Using device: {self.device}')

    def train(self):
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_name).to(self.device)

        seq2seq_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

        train_ds = load_from_disk(str(self.config.transformed_train_dir))
        test_ds = load_from_disk(str(self.config.transformed_test_dir))

        training_args = TrainingArguments(
            output_dir=str(self.config.root_dir),
            num_train_epochs=self.config.num_train_epochs,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_eval_batch_size,
            warmup_steps=self.config.warmup_steps,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            evaluation_strategy=self.config.evaluation_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            fp16=torch.cuda.is_available(),
            report_to="none"
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            test_dataset=test_ds,
            eval_dataset=test_ds,
            data_collator=seq2seq_collator,
            tokenizer=tokenizer,
        )

        print("Starting training...")
        trainer.train()

        ## Save the final model
        model.save_pretrained(str(self.config.root_dir / "final_model"))
        tokenizer.save_pretrained(str(self.config.root_dir / "tokenizer"))
        print(f'Model and tokenizer saved to: {self.config.root_dir}')

c:\Users\HP\Text-Summarizer\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

     ------------------------------------ 374.9/374.9 KB 457.6 kB/s eta 0:00:00


You should consider upgrading via the 'C:\Users\HP\Text-Summarizer\venv\Scripts\python.exe -m pip install --upgrade pip' command.


Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: accelerate 1.10.1
Uninstalling accelerate-1.10.1:
  Successfully uninstalled accelerate-1.10.1

You should consider upgrading via the 'C:\Users\HP\Text-Summarizer\venv\Scripts\python.exe -m pip install --upgrade pip' command.



  Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
  Using cached accelerate-1.10.1-py3-none-any.whl (374 kB)


### Run the Pipeline

In [ ]:
try:
    config_manager = ConfigurationManager()
    trainer_config = config_manager.get_model_trainer_config()
    model_trainer = ModelTrainer(trainer_config)
    model_trainer.train()
except Exception as e:
    raise e